In [7]:
import asyncio
import json
import time
from typing import List, Dict, Any

In [8]:
# --- Step 1: Define Specialized Execution Tools (Simulated Network Latency) ---

async def execute_patch_binary_tool(binary_offset: str, hex_bytes: str) -> Dict[str, Any]:
    """Simulates a low-level binary modification tool."""
    await asyncio.sleep(0.5) # Simulate minor disk I/O lag
    print(f"  [EXECUTION] Modifying memory offset {binary_offset} with patch data: {hex_bytes}...")
    return {"status": "PATCH_SUCCESSFUL", "offset": binary_offset}

async def execute_remote_server_ping_tool(ip_address: str) -> Dict[str, Any]:
    """Simulates a fragile network check tool that intentionally crashes on specific IPs."""
    await asyncio.sleep(0.8) # Simulate network ping delay
    print(f"  [EXECUTION] Pinging remote system infrastructure node at: {ip_address}...")
    
    # We intentionally simulate a hard system connection crash for a specific IP address target
    if ip_address == "10.0.0.45":
        raise ConnectionRefusedError("CRITICAL_ERROR: Remote server refused connections on port 22.")
        
    return {"status": "ONLINE", "latency_ms": 14}

In [9]:
# --- Step 2: The Parallel & Resilient Agent Loop Driver ---

class ResilientParallelAgentEngine:
    def __init__(self):
        print("[ENGINE] Asynchronous multi-thread tool controller initialized.")

    async def run_parallel_turn(self, tool_requests: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Executes an array of concurrent tool operations concurrently 
        while explicitly isolating failures.
        """
        tasks = []
        
        # Build our execution pool
        for request in tool_requests:
            name = request["name"]
            args = request["args"]
            call_id = request["id"]
            
            if name == "patch_binary":
                tasks.append(self._wrap_tool_execution(call_id, name, execute_patch_binary_tool(**args)))
            elif name == "ping_server":
                tasks.append(self._wrap_tool_execution(call_id, name, execute_remote_server_ping_tool(**args)))

        print(f"\n[ENGINE] Dispatching {len(tasks)} tool operations concurrently via asyncio...")
        start_time = time.time()
        
        # PITFALL WARNING: We use asyncio.gather to run tasks in parallel.
        # This collapses total time from linear accumulation down to the single slowest task.
        completed_tool_responses = await asyncio.gather(*tasks)
        
        duration = time.time() - start_time
        print(f"[ENGINE] Concurrent batch completed execution in {duration:.2f} seconds.")
        return completed_tool_responses

    async def _wrap_tool_execution(self, call_id: str, name: str, coroutine) -> Dict[str, Any]:
        """Secures individual task execution channels against unhandled app-crashing bugs."""
        try:
            # Attempt to execute the raw asynchronous operation block
            result = await coroutine
            return {
                "role": "tool",
                "tool_call_id": call_id,
                "name": name,
                "content": json.dumps(result)
            }
        except Exception as crash_error:
            # PITFALL WARNING: If the tool fails, do NOT crash the program!
            # Intercept the exception and format it cleanly so the LLM can try to fix it.
            print(f"  [RECOVERY ACTIVATED] Caught tool failure inside '{name}' (ID: {call_id}).")
            return {
                "role": "tool",
                "tool_call_id": call_id,
                "name": name,
                "content": f"TOOL_EXECUTION_FAILED_ERROR: {str(crash_error)}"
            }

In [10]:
# --- Step 3: Run the Asynchronous Pipeline ---

async def main():
    agent = ResilientParallelAgentEngine()
    
    # Simulating a structural payload returned by an LLM that wants to perform 
    # two server checks and a binary code patch at the exact same time
    simulated_parallel_llm_calls = [
        {"id": "call_01", "name": "ping_server", "args": {"ip_address": "192.168.1.10"}},
        {"id": "call_02", "name": "ping_server", "args": {"ip_address": "10.0.0.45"}}, # This one will blow up
        {"id": "call_03", "name": "patch_binary", "args": {"binary_offset": "0x0040A1F0", "hex_bytes": "90 90"}}
    ]
    
    final_payload_track = await agent.run_parallel_turn(simulated_parallel_llm_calls)
    
    print("\n" + "="*70)
    print("STRUCTURAL RESULT RETURNED TO THE MODEL'S REASONING CONTEXT:")
    print("="*70)
    print(json.dumps(final_payload_track, indent=2))

In [11]:
# Run the async initialization block
await main()

[ENGINE] Asynchronous multi-thread tool controller initialized.

[ENGINE] Dispatching 3 tool operations concurrently via asyncio...
  [EXECUTION] Modifying memory offset 0x0040A1F0 with patch data: 90 90...
  [EXECUTION] Pinging remote system infrastructure node at: 192.168.1.10...
  [EXECUTION] Pinging remote system infrastructure node at: 10.0.0.45...
  [RECOVERY ACTIVATED] Caught tool failure inside 'ping_server' (ID: call_02).
[ENGINE] Concurrent batch completed execution in 0.80 seconds.

STRUCTURAL RESULT RETURNED TO THE MODEL'S REASONING CONTEXT:
[
  {
    "role": "tool",
    "tool_call_id": "call_01",
    "name": "ping_server",
    "content": "{\"status\": \"ONLINE\", \"latency_ms\": 14}"
  },
  {
    "role": "tool",
    "tool_call_id": "call_02",
    "name": "ping_server",
    "content": "TOOL_EXECUTION_FAILED_ERROR: CRITICAL_ERROR: Remote server refused connections on port 22."
  },
  {
    "role": "tool",
    "tool_call_id": "call_03",
    "name": "patch_binary",
    "conten